# 04: Spatial Data & Census Boundaries

This notebook demonstrates the spatial data capabilities of siege_utilities, including:

1. **Census Boundary Downloads** - Download TIGER/Line shapefiles
2. **State FIPS Normalization** - Convert between state names, abbreviations, and FIPS codes
3. **Geographic Level Discovery** - Find available boundary types for any year
4. **Working with GeoDataFrames** - Basic spatial operations

## Prerequisites

```bash
pip install -e /path/to/siege_utilities
```

## What this shows
Fetching US Census TIGER boundaries (states, counties, tracts) via the BoundaryProvider registry.

## Why it matters
Canonical entry point for geographic work; exercises the `geo/providers/` consolidation from ELE-2438.

## Prereqs
- `pip install 'siege-utilities[geo]'`

## Next
- `05_Choropleth_Maps.ipynb`, `07_Geocoding_Address_Processing.ipynb`, `23_Redistricting_Analysis.ipynb`.


> **This is a comprehensive reference notebook** covering 9+ spatial subsystems. For focused introductions, see:
> - **NB05** — Choropleth maps and classification
> - **NB07** — Geocoding and address processing
> - **NB15** — Census demographics quick start
> - **NB25** — SpatiaLite cache and isochrones
> - **NB26** — International boundaries (GADM)


In [ ]:
import siege_utilities as su

# Initialize logging
su.configure_shared_logging(level="INFO")

In [ ]:
%matplotlib inline

# Imports
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from siege_utilities.geo.spatial_data import (
    # Census data functions
    get_census_boundaries,
    download_data,
    get_available_years,
    get_year_directory_contents,
    discover_boundary_types,
    
    # Reference data
    BOUNDARY_TYPE_CATALOG,
    
    # State normalization
    normalize_state_identifier,
    get_state_by_abbreviation,
    get_state_by_name,
    
    # Global instances
    census_source
)

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

su.log_info("Imports successful!")

# --- Output configuration ---
OUTPUT_DIR = Path("output") / "notebook_04"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

su.log_info(f"Output directory: {OUTPUT_DIR}  (exists={OUTPUT_DIR.exists()})")

In [ ]:
# Retrieve Census API key: try 1Password first, fall back to env var
from siege_utilities.config.credential_manager import get_credential
import os

_key_source = None
census_api_key = get_credential(
    service='Census API Credentials',
    username='api',
    field='credential',
    vault='Employee',
    account='Siege_Analytics'
)
if census_api_key:
    _key_source = '1Password (Employee/Census API Credentials)'
else:
    census_api_key = os.environ.get('CENSUS_API_KEY')
    if census_api_key:
        _key_source = 'environment variable (CENSUS_API_KEY)'

if census_api_key:
    os.environ['CENSUS_API_KEY'] = census_api_key
    su.log_info(f"Census API key loaded from {_key_source}")
else:
    su.log_warning("Census API key not found — set CENSUS_API_KEY env var or sign into 1Password")

## 1. State FIPS Normalization

The `normalize_state_identifier()` function accepts any state format and returns the FIPS code.

In [ ]:
# Test various input formats
test_inputs = [
    '06',           # FIPS code
    'CA',           # Abbreviation
    'California',   # Full name
    'california',   # Lowercase
    'CALIFORNIA',   # Uppercase
    'TX',
    'New York',
    'ny',
]

su.log_info("State FIPS Normalization")
su.log_info("=" * 40)
for inp in test_inputs:
    fips = normalize_state_identifier(inp)
    su.log_info(f"{inp:15} -> {fips}")

In [ ]:
# Get full state info
ca_info = get_state_by_abbreviation('CA')
su.log_info("State info for CA:")
su.log_info(str(ca_info))

su.log_info("")
tx_info = get_state_by_name('Texas')
su.log_info("State info for Texas:")
su.log_info(str(tx_info))

## 2. Discovering Available Census Data

Census TIGER/Line files are available for different years. Let's discover what's available.

In [ ]:
# Get available years
years = get_available_years()
su.log_info(f"Available Census years: {len(years)} years")
su.log_info(f"Range: {min(years)} - {max(years)}")
su.log_info(f"Recent years: {sorted(years)[-5:]}")

In [ ]:
# Discover boundary types for 2020
boundary_types = discover_boundary_types(2020)
su.log_info(f"Boundary types available for 2020: {len(boundary_types)}")

# Display available types grouped by category using the module catalog
for category, label in [('redistricting', 'Redistricting & Electoral'), ('general', 'General')]:
    available = {k: v for k, v in BOUNDARY_TYPE_CATALOG.items()
                 if v['category'] == category and k in boundary_types}
    su.log_info(f"\n{label} ({len(available)} available):")
    for key, info in available.items():
        su.log_info(f"  {info['abbrev']:12s}  {key:18s}  {info['name']}")

# Flag any types returned by Census not in our catalog
uncatalogued = set(boundary_types.keys()) - set(BOUNDARY_TYPE_CATALOG.keys())
if uncatalogued:
    su.log_info(f"\nUncatalogued types: {sorted(uncatalogued)}")

## 3. Downloading Census Boundaries

### 3.1 State Boundaries (National)

In [ ]:
# Download all US state boundaries
states_gdf = get_census_boundaries(
    year=2020,
    geographic_level='state'
)

su.log_info(f"Downloaded {len(states_gdf)} states/territories")
su.log_info(f"Columns: {list(states_gdf.columns)}")
su.log_info(f"CRS: {states_gdf.crs}")

In [ ]:
# Quick visualization of continental US
continental = states_gdf[
    ~states_gdf['statefp'].isin(['02', '15', '72', '78', '60', '66', '69'])
]

fig, ax = plt.subplots(1, 1, figsize=(12, 8))
continental.plot(ax=ax, edgecolor='black', facecolor='lightblue', linewidth=0.5)
ax.set_title('Continental United States')
ax.axis('off')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'continental_us.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()

### 3.2 County Boundaries (State-specific)

In [ ]:
# Download county boundaries for California
ca_counties = get_census_boundaries(
    year=2020,
    geographic_level='county',
    state_fips='06'  # California
)

# Filter to just CA counties (national file includes all)
ca_counties = ca_counties[ca_counties['statefp'] == '06']

su.log_info(f"California has {len(ca_counties)} counties")
su.log_info("Sample counties:")
su.log_info(str(ca_counties[['name', 'geoid', 'aland']].head(10)))

In [ ]:
# Visualize California counties
fig, ax = plt.subplots(1, 1, figsize=(10, 12))
ca_counties.plot(ax=ax, edgecolor='black', facecolor='lightgreen', linewidth=0.5)
ax.set_title('California Counties')
ax.axis('off')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'california_counties.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()

### 3.3 Census Tracts (County-specific)

In [ ]:
# Download census tracts for Los Angeles County
la_tracts = get_census_boundaries(
    year=2020,
    geographic_level='tract',
    state_fips='06'  # California
)

# Filter to LA County (FIPS 037)
la_tracts = la_tracts[la_tracts['countyfp'] == '037']

su.log_info(f"Los Angeles County has {len(la_tracts)} census tracts")
su.log_info(f"Geoid format: {la_tracts['geoid'].iloc[0]}")

In [ ]:
# Visualize LA County tracts
fig, ax = plt.subplots(1, 1, figsize=(12, 10))
la_tracts.plot(ax=ax, edgecolor='gray', facecolor='lightyellow', linewidth=0.1)
ax.set_title(f'Los Angeles County Census Tracts ({len(la_tracts)} tracts)')
ax.axis('off')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'la_county_tracts.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()

## 4. Working with Geographic Data

### 4.1 Calculate Areas

In [ ]:
# Calculate area in square miles from ALAND (land area in sq meters)
ca_counties['area_sq_mi'] = ca_counties['aland'] / 2_589_988  # sq meters to sq miles

su.log_info("Largest California counties by area:")
su.log_info(str(ca_counties.nlargest(5, 'area_sq_mi')[['name', 'area_sq_mi']]))

su.log_info("Smallest California counties by area:")
su.log_info(str(ca_counties.nsmallest(5, 'area_sq_mi')[['name', 'area_sq_mi']]))

### 4.2 Spatial Joins

In [ ]:
# Create sample point data (cities in CA)
from shapely.geometry import Point

cities = gpd.GeoDataFrame({
    'city': ['Los Angeles', 'San Francisco', 'San Diego', 'Sacramento', 'Fresno'],
    'geometry': [
        Point(-118.2437, 34.0522),
        Point(-122.4194, 37.7749),
        Point(-117.1611, 32.7157),
        Point(-121.4944, 38.5816),
        Point(-119.7871, 36.7378)
    ]
}, crs='EPSG:4326')

# Spatial join to find which county each city is in
cities_with_county = gpd.sjoin(cities, ca_counties[['name', 'geometry']], how='left', predicate='within')
cities_with_county = cities_with_county.rename(columns={'name': 'county'})

su.log_info("Cities with their counties:")
su.log_info(str(cities_with_county[['city', 'county']]))

## 5. Alternative Download Function

The `download_data()` function provides a simpler interface.

In [ ]:
# Using download_data() - equivalent functionality
tx_counties = download_data(
    year=2020,
    geographic_level='county',
    state_fips='48'  # Texas
)

tx_counties = tx_counties[tx_counties['statefp'] == '48']
su.log_info(f"Texas has {len(tx_counties)} counties")

## Summary (Part 1: Boundaries)

This section demonstrated:

| Function | Purpose |
|----------|----------|
| `normalize_state_identifier()` | Convert any state format to FIPS |
| `get_available_years()` | List available Census years |
| `discover_boundary_types()` | Find boundary types for a year |
| `get_census_boundaries()` | Download TIGER/Line shapefiles |
| `download_data()` | Simplified download interface |

**Key Parameters:**
- `year`: Census year (1992-present)
- `geographic_level`: state, county, tract, bg (block group), place, zcta, cd
- `state_fips`: Required for tract, bg, block levels

---

## Part 2: Census Demographics (CensusAPIClient)

The `CensusAPIClient` fetches demographic data from the Census Bureau API (ACS, Decennial).
It complements boundary downloads by providing actual demographic values.

In [ ]:
# Import CensusAPIClient
import os
from siege_utilities.geo.census_api_client import (
    CensusAPIClient,
    get_demographics,
    get_population,
    get_income_data,
    get_census_data_with_geometry,
    VARIABLE_GROUPS
)

# Show available variable groups
su.log_info("Available predefined variable groups:")
for group_name, variables in VARIABLE_GROUPS.items():
    su.log_info(f"  {group_name}: {len(variables)} variables")

### Quick Demographic Fetch (Convenience Functions)

These functions work without an API key for basic queries (rate-limited).

In [ ]:
# Get population data for California counties (2022 ACS 5-year)
# Note: API key optional but recommended for higher rate limits
# Set CENSUS_API_KEY environment variable or pass api_key parameter

try:
    ca_pop = get_population(
        state='California',
        geography='county',
        year=2022
    )
    su.log_info(f"Retrieved population for {len(ca_pop)} California counties")
    su.log_info("Top 5 most populous counties:")
    su.log_info(str(ca_pop.nlargest(5, 'B01001_001E')[['NAME', 'B01001_001E']]))
except Exception as e:
    su.log_warning(f"Note: Census API may require an API key. Error: {e}")
    su.log_info("Get a free key at: https://api.census.gov/data/key_signup.html")

### Combined Geometry + Demographics

The most powerful function: download boundaries AND demographics in one call.

In [ ]:
# Get California counties with income data - geometry + demographics in one GeoDataFrame
try:
    ca_income_gdf = get_census_data_with_geometry(
        year=2022,
        geography='county',
        state_fips='06',  # California
        variables=['B19013_001E']  # Median household income
    )
    
    su.log_info(f"GeoDataFrame with {len(ca_income_gdf)} counties")
    su.log_info(f"Columns: {list(ca_income_gdf.columns)}")
    
    # Create a choropleth map of median income
    fig, ax = plt.subplots(1, 1, figsize=(10, 12))
    ca_income_gdf.plot(
        column='B19013_001E',
        ax=ax,
        legend=True,
        legend_kwds={'label': 'Median Household Income ($)'},
        cmap='YlGnBu',
        edgecolor='black',
        linewidth=0.3
    )
    ax.set_title('California Counties - Median Household Income (2022 ACS)')
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'ca_income_choropleth.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()
    
except Exception as e:
    su.log_warning(f"Note: Requires Census API. Error: {e}")

### GEOID Utilities

GEOIDs are standardized identifiers for Census geographies. These utilities help with normalization and joining.

In [ ]:
from siege_utilities.geo.geoid_utils import (
    construct_geoid,
    parse_geoid,
    normalize_geoid,
    validate_geoid,
    can_normalize_geoid,
    GEOID_LENGTHS
)

# GEOID lengths by geography type
su.log_info("GEOID lengths by geography:")
for geo, length in GEOID_LENGTHS.items():
    su.log_info(f"  {geo}: {length} digits")

su.log_info("=" * 50)

# Construct GEOIDs from components
tract_geoid = construct_geoid(
    geography='tract',
    state='06',      # California
    county='037',    # Los Angeles
    tract='207400'   # Tract number
)
su.log_info(f"Constructed tract GEOID: {tract_geoid}")

# Parse a GEOID back to components
components = parse_geoid(tract_geoid, 'tract')
su.log_info(f"Parsed components: {components}")

# Normalize incomplete GEOIDs (adds leading zeros)
su.log_info(f"Normalize '6037' as county: {normalize_geoid('6037', 'county')}")
su.log_info(f"Normalize '6' as state: {normalize_geoid('6', 'state')}")

# Validate GEOIDs — strict by default (requires proper zero-padding)
su.log_info(f"Is '06037' a valid county GEOID? {validate_geoid('06037', 'county')}")
su.log_info(f"Is '6037' a valid county GEOID? {validate_geoid('6037', 'county')}")

# Check normalizability — can a value be zero-padded to a valid GEOID?
su.log_info(f"Can '6037' be normalized to a county GEOID? {can_normalize_geoid('6037', 'county')}")
su.log_info(f"Can 'CA037' be normalized to a county GEOID? {can_normalize_geoid('CA037', 'county')}")

## Part 4: Intelligent Dataset Selection

The `CensusDataSelector` helps you choose the right Census dataset for your analysis.
Given an analysis type, geography level, and time period, it recommends which datasets
to use and checks compatibility.

In [ ]:
from siege_utilities.geo.census_data_selector import (
    CensusDataSelector,
    select_datasets_for_analysis,
    get_dataset_compatibility_matrix,
    suggest_analysis_approach,
)

# Initialize selector
selector = CensusDataSelector()

# Example 1: What datasets work for poverty analysis at tract level?
result = select_datasets_for_analysis(
    analysis_type='poverty',
    geography_level='tract',
    time_period='2015-2022'
)
su.log_info("Dataset selection for poverty analysis at tract level:")
for key, val in result.items():
    su.log_info(f"  {key}: {val}")

# Example 2: Compatibility matrix
matrix = get_dataset_compatibility_matrix()
su.log_info(f"Compatibility matrix shape: {matrix.shape}")
display(matrix)

# Example 3: Suggest an approach
approach = suggest_analysis_approach(
    analysis_type='demographics',
    geography_level='county',
    time_constraints='standard'
)
su.log_info("Suggested approach:")
for key, val in approach.items():
    su.log_info(f"  {key}: {val}")

In [ ]:
# Visualize the compatibility matrix as a heatmap
fig, ax = plt.subplots(figsize=(10, 6))
matrix_plot = matrix.set_index('analysis_type') if 'analysis_type' in matrix.columns else matrix
im = ax.imshow(matrix_plot.values, cmap='YlGnBu', aspect='auto')

ax.set_xticks(range(len(matrix_plot.columns)))
ax.set_xticklabels(matrix_plot.columns, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(matrix_plot.index)))
ax.set_yticklabels(matrix_plot.index, fontsize=9)

# Add score values in each cell
for i in range(len(matrix_plot.index)):
    for j in range(len(matrix_plot.columns)):
        val = matrix_plot.iloc[i, j]
        color = 'white' if val > 0.6 else 'black'
        ax.text(j, i, f'{val:.1f}', ha='center', va='center', color=color, fontsize=8)

plt.colorbar(im, ax=ax, label='Compatibility Score', shrink=0.8)
ax.set_title('Census Dataset Compatibility by Analysis Type')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'dataset_compatibility_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()

## Part 5: Boundary Crosswalks

Census boundaries change between decennial censuses. Crosswalk files map geographies
from one vintage to another (e.g., 2010 tracts to 2020 tracts), enabling longitudinal analysis.

In [ ]:
from siege_utilities.geo.crosswalk import (
    CrosswalkClient,
    get_crosswalk,
    get_crosswalk_metadata,
    list_available_crosswalks,
)
from siege_utilities.geo.crosswalk.relationship_types import (
    RelationshipType, WeightMethod, CrosswalkMetadata
)

# List what's available
available = list_available_crosswalks()
su.log_info(f"Available crosswalks: {len(available)}")
for cw in available:
    su.log_info(f"  {cw}")

# Get tract crosswalk metadata (2010 -> 2020)
try:
    meta = get_crosswalk_metadata(source_year=2010, target_year=2020, geography_level='tract')
    su.log_info(f"Tract crosswalk 2010->2020:")
    for key, val in meta.summary().items():
        su.log_info(f"  {key}: {val}")
except Exception as e:
    su.log_warning(f"Crosswalk metadata not available: {e}")
    su.log_info("This requires downloading Census crosswalk files")

## Part 6: Time-Series Analysis

Analyze longitudinal Census data with change metrics and trend classification.
This section includes dual-mode cells (Pandas + optional Spark).

In [ ]:
# Spark availability (for dual-mode cells)
SPARK_AVAILABLE = False
spark = None
try:
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F
    SPARK_AVAILABLE = True
    su.log_info("PySpark available for dual-mode cells")
except ImportError:
    su.log_info("PySpark not installed -- pandas-only mode")

In [ ]:
import numpy as np

from siege_utilities.geo.timeseries.change_metrics import (
    calculate_change_metrics,
    calculate_index,
    get_change_summary,
)
from siege_utilities.geo.timeseries.trend_classifier import (
    classify_trends,
    TrendThresholds,
    TrendCategory,
    get_trend_summary,
    identify_outliers,
)

# Create sample longitudinal data
np.random.seed(42)
n_tracts = 50
longitudinal_df = pd.DataFrame({
    'GEOID': [f'0603700{i:04d}' for i in range(n_tracts)],
    'income_2010': np.random.normal(60000, 15000, n_tracts).astype(int),
    'income_2015': np.random.normal(65000, 16000, n_tracts).astype(int),
    'income_2020': np.random.normal(72000, 18000, n_tracts).astype(int),
})

su.log_info(f"Longitudinal data: {longitudinal_df.shape}")
display(longitudinal_df.head())

# Calculate change metrics between 2010 and 2020
longitudinal_df['pct_change'] = (
    (longitudinal_df['income_2020'] - longitudinal_df['income_2010']) / 
    longitudinal_df['income_2010'] * 100
)

# Classify trends
classified = classify_trends(
    longitudinal_df, 'pct_change',
    thresholds=TrendThresholds(rapid_growth=25, moderate_growth=10, stable_lower=-10, moderate_decline=-25)
)
su.log_info("Trend classification:")
su.log_info(str(classified['trend_category'].value_counts()))

# Identify outliers
outliers = identify_outliers(longitudinal_df, 'pct_change', method='iqr')
n_outliers = outliers['is_outlier'].sum()
su.log_info(f"Outliers detected: {n_outliers} of {len(outliers)}")

# Summary
summary = get_trend_summary(classified, trend_column='trend_category')
su.log_info("Trend summary:")
for key, val in summary.items():
    su.log_info(f"  {key}: {val}")

In [ ]:
# Trend category distribution
trend_counts = classified['trend_category'].value_counts()

# Color-code by trend direction
trend_colors = {
    'rapid_growth': '#1a9641',
    'moderate_growth': '#a6d96a',
    'stable': '#bdbdbd',
    'moderate_decline': '#fdae61',
    'rapid_decline': '#d7191c',
}
colors = [trend_colors.get(str(cat), '#999999') for cat in trend_counts.index]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(len(trend_counts)), trend_counts.values, color=colors, edgecolor='black', linewidth=0.5)
ax.set_xticks(range(len(trend_counts)))
ax.set_xticklabels([str(c).replace('_', ' ').title() for c in trend_counts.index], rotation=30, ha='right')
ax.set_ylabel('Number of Tracts')
ax.set_title('Income Trend Classification (2010-2020)')

# Add count labels on bars
for bar, count in zip(bars, trend_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5, str(count),
            ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'trend_categories.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
# Percent change distribution with trend threshold boundaries
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(longitudinal_df['pct_change'], bins=20, color='#4393c3', edgecolor='black', linewidth=0.5, alpha=0.8)

# Draw threshold boundaries
thresholds = {'Rapid Decline': -25, 'Moderate Decline': -10, 'Moderate Growth': 10, 'Rapid Growth': 25}
for label, val in thresholds.items():
    ax.axvline(val, color='#d62728', linestyle='--', linewidth=1.5, alpha=0.8)
    ax.text(val, ax.get_ylim()[1] * 0.92, f' {label}\n ({val:+d}%)',
            fontsize=7, rotation=0, va='top', ha='center',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8, edgecolor='gray'))

ax.set_xlabel('Percent Change in Income (2010-2020)')
ax.set_ylabel('Number of Tracts')
ax.set_title('Distribution of Income Change with Trend Classification Boundaries')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pct_change_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
# Dual-mode: trend aggregation
# Pandas (always runs)
pdf_trend_agg = classified.groupby('trend_category').agg(
    count=('GEOID', 'count'),
    avg_change=('pct_change', 'mean')
).reset_index()
su.log_info("Pandas -- Trend aggregation:")
display(pdf_trend_agg)

# Spark (conditional)
if SPARK_AVAILABLE and spark:
    sdf = spark.createDataFrame(classified[['GEOID', 'pct_change', 'trend_category']])
    sdf_agg = sdf.groupBy('trend_category').agg(
        F.count('GEOID').alias('count'),
        F.avg('pct_change').alias('avg_change')
    )
    su.log_info("Spark -- Trend aggregation:")
    sdf_agg.show()

## Part 7: Isochrones and 3D Geo Visualization

This section demonstrates:

- building open-source isochrone requests (ORS/Valhalla)
- optional live isochrone fetch when an API key is available
- rendering a 3D map from city-level geo points


In [ ]:
import os
import json
import pandas as pd

from siege_utilities.geo.isochrones import build_isochrone_request, get_isochrone
from siege_utilities.reporting import ChartGenerator, save_rl_image, show_rl_image

# Use Los Angeles as an example center point
la_point = cities[cities['city'] == 'Los Angeles'].iloc[0].geometry
lat, lon = float(la_point.y), float(la_point.x)

# 1) Build (provider-agnostic) isochrone request definition
ors_request = build_isochrone_request(
    latitude=lat,
    longitude=lon,
    travel_time_minutes=15,
    provider='openrouteservice',
    profile='driving-car',
)

valhalla_request = build_isochrone_request(
    latitude=lat,
    longitude=lon,
    travel_time_minutes=20,
    provider='valhalla',
    base_url='http://valhalla.svc.cluster.local:8002',
    profile='driving-car',
)

print('ORS request URL:', ors_request['url'])
print('Valhalla request URL:', valhalla_request['url'])

# 2) Optional live ORS call (skips cleanly if ORS_API_KEY is not set)
ors_api_key = os.getenv('ORS_API_KEY')
if ors_api_key:
    try:
        iso_geojson = get_isochrone(
            latitude=lat,
            longitude=lon,
            travel_time_minutes=15,
            provider='openrouteservice',
            api_key=ors_api_key,
            profile='driving-car',
        )
        with open(OUTPUT_DIR / 'la_isochrone_15min.geojson', 'w', encoding='utf-8') as f:
            json.dump(iso_geojson, f)
        print('Saved:', OUTPUT_DIR / 'la_isochrone_15min.geojson')
    except Exception as e:
        print('Live ORS isochrone request failed:', type(e).__name__, str(e))
else:
    print('ORS_API_KEY not set; skipping live network call.')

# 3) 3D map demo from city points
city_3d = pd.DataFrame({
    'name': cities['city'].tolist(),
    'lat': [geom.y for geom in cities.geometry],
    'lon': [geom.x for geom in cities.geometry],
    'value': [120, 95, 80, 60, 70],
})

chart_gen_geo = ChartGenerator(output_dir=OUTPUT_DIR)
img_3d = chart_gen_geo.create_3d_map(
    city_3d,
    latitude_column='lat',
    longitude_column='lon',
    elevation_column='value',
    title='California City 3D Intensity Demo',
)

save_rl_image(img_3d, OUTPUT_DIR / 'geo_3d_city_intensity.png')
display(show_rl_image(img_3d))


## Part 8: H3 Hexagonal Spatial Indexing

H3 is Uber's hexagonal hierarchical spatial index. The `geo.h3_utils` module
provides lightweight spatial join approximation using H3 — a "geo-lite" fallback
when full GeoPandas `sjoin` operations are too expensive.

Key functions:

- `h3_index_points(df, lat_col, lon_col, resolution)` — assign H3 hex IDs to point rows
- `h3_index_polygon(geometry, resolution)` — get hex IDs covering a polygon
- `h3_spatial_join(points_df, polygons_gdf, ...)` — approximate point-in-polygon via hex overlap
- `h3_hex_to_boundary(hex_id)` — get boundary coordinates for a hex cell
- `h3_resolution_for_area(target_km2)` — suggest resolution for a target area

**When to use:**
- Millions of point-in-polygon joins where GeoPandas `sjoin` is too slow
- Spatial aggregation at multiple resolutions
- Pre-indexing data for fast lookups in Spark or DuckDB

In [ ]:
try:
    from siege_utilities.geo.h3_utils import (
        h3_index_points,
        h3_index_polygon,
        h3_spatial_join,
        h3_hex_to_boundary,
        h3_resolution_for_area,
        H3_AVAILABLE,
        _H3_RESOLUTION_AREA_KM2,
    )
    HAS_H3_MODULE = True
except ImportError as e:
    HAS_H3_MODULE = False
    su.log_info(f"h3_utils import failed ({e}).")

if HAS_H3_MODULE:
    su.log_info("H3 Resolution Reference (approximate hex areas):")
    su.log_info(f"{'Res':>4}  {'Area (km2)':>14}  {'~Description'}")
    su.log_info("-" * 50)
    descriptions = {
        0: "Continent", 1: "Large country", 2: "Large state",
        3: "US state", 4: "Large county", 5: "City metro",
        6: "Small city", 7: "Large neighbourhood", 8: "Neighbourhood",
        9: "City block", 10: "Sub-block", 11: "Building",
        12: "Small building", 13: "Room", 14: "Desk", 15: "Tiny",
    }
    for res, area in _H3_RESOLUTION_AREA_KM2.items():
        su.log_info(f"{res:>4}  {area:>14,.3f}  {descriptions.get(res, '')}")

    su.log_info(f"\nh3 library available: {H3_AVAILABLE}")
else:
    su.log_info("h3_utils module not available.")

In [ ]:
if HAS_H3_MODULE and H3_AVAILABLE:
    # Index the California cities we created earlier using H3
    cities_h3 = pd.DataFrame({
        "city": cities["city"].tolist(),
        "latitude": [geom.y for geom in cities.geometry],
        "longitude": [geom.x for geom in cities.geometry],
    })

    hex_ids = h3_index_points(cities_h3, "latitude", "longitude", resolution=8)
    cities_h3["h3_index"] = hex_ids

    su.log_info("H3-indexed California cities (resolution 8):")
    su.log_info(cities_h3[["city", "h3_index"]].to_string(index=False))

    # Get boundary vertices for the first hex
    first_hex = hex_ids.iloc[0]
    boundary = h3_hex_to_boundary(first_hex)
    su.log_info(f"\nBoundary vertices for {first_hex}: {len(boundary)} points")
    for lat, lng in boundary[:3]:
        su.log_info(f"  ({lat:.6f}, {lng:.6f})")
    su.log_info("  ...")

    # Resolution suggestions for different target areas
    su.log_info("\nResolution suggestions:")
    for target_km2 in [1.0, 10.0, 100.0, 1000.0]:
        res = h3_resolution_for_area(target_km2)
        actual = _H3_RESOLUTION_AREA_KM2[res]
        su.log_info(f"  Target {target_km2:>7,.1f} km2 -> resolution {res} (actual {actual:,.3f} km2)")

elif HAS_H3_MODULE:
    su.log_info("h3 library not installed. Install with:")
    su.log_info("  pip install 'siege-utilities[h3]'")

## Part 9: Boundary Providers

The `geo.boundary_providers` module provides a pluggable architecture for
fetching administrative boundary geometries from different sources:

- `BoundaryProvider` (ABC) — with `get_boundary()`, `list_levels()`, `is_available()`
- `CensusTIGERProvider` — US Census TIGER/Line shapefiles (wraps `get_census_boundaries()`)
- `GADMProvider` — international boundaries via GADM
- `resolve_boundary_provider(country)` — factory that picks TIGER for US, GADM otherwise

This complements the boundary download functions from Part 1 by adding a
provider abstraction. Instead of calling `get_census_boundaries()` directly,
you can use `resolve_boundary_provider()` to get the right provider for any
country, then call a uniform API.

**When to use:**
- Building geographic base layers for choropleths
- Supporting both US and international campaigns from a single API
- Fetching county, tract, or admin boundaries on demand without
  knowing which data source to use

In [ ]:
from siege_utilities.geo.boundary_providers import (
    BoundaryProvider,
    CensusTIGERProvider,
    GADMProvider,
    resolve_boundary_provider,
)

# Resolve providers for different countries
us_provider = resolve_boundary_provider("US")
de_provider = resolve_boundary_provider("DE")
pr_provider = resolve_boundary_provider("PR")

su.log_info("Provider Resolution:")
for country, prov in [("US", us_provider), ("DE", de_provider), ("PR", pr_provider)]:
    su.log_info(f"  {country} -> {prov.provider_name} (available: {prov.is_available()})")

# Inspect available geographic levels for each provider
tiger = CensusTIGERProvider()
su.log_info(f"\nCensus TIGER levels: {tiger.list_levels()}")

gadm = GADMProvider()
su.log_info(f"GADM levels: {gadm.list_levels()}")
su.log_info(f"GADM available (requires geopandas): {gadm.is_available()}")

## Complete Summary

### Part 1: Boundary Downloads

| Function | Purpose |
|----------|--------|
| `get_census_boundaries()` | Download TIGER/Line shapefiles |
| `normalize_state_identifier()` | Convert state names/abbrevs to FIPS |
| `get_available_years()` | List available Census years |
| `discover_boundary_types()` | Find boundary types for a year |

### Part 2: Demographics (CensusAPIClient)

| Function | Purpose |
|----------|--------|
| `get_population()` | Fetch population counts |
| `get_demographics()` | Fetch demographic variables |
| `get_income_data()` | Fetch income-related variables |
| `get_census_data_with_geometry()` | Combined: boundaries + demographics in one GeoDataFrame |

### Part 3: GEOID Utilities

| Function | Purpose |
|----------|--------|
| `construct_geoid()` | Build GEOID from components |
| `parse_geoid()` | Extract components from GEOID |
| `normalize_geoid()` | Add leading zeros |
| `validate_geoid()` | Check GEOID format (strict -- requires leading zeros) |
| `can_normalize_geoid()` | Check if a value can be zero-padded to a valid GEOID |

### Part 4: Intelligent Dataset Selection

| Function | Purpose |
|----------|--------|
| `CensusDataSelector()` | Initialize dataset selector |
| `select_datasets_for_analysis()` | Recommend datasets for an analysis type/geography/time period |
| `get_dataset_compatibility_matrix()` | Show which datasets support which geographies |
| `suggest_analysis_approach()` | Get a recommended approach for your analysis |

### Part 5: Boundary Crosswalks

| Function | Purpose |
|----------|--------|
| `list_available_crosswalks()` | List available crosswalk files |
| `get_crosswalk()` | Download a crosswalk table |
| `get_crosswalk_metadata()` | Get metadata about a crosswalk |
| `CrosswalkClient` | Full client for crosswalk operations |

### Part 6: Time-Series Analysis

| Function | Purpose |
|----------|--------|
| `classify_trends()` | Classify percent-change values into trend categories |
| `TrendThresholds` | Configure thresholds for trend classification |
| `identify_outliers()` | Detect outliers using IQR or z-score methods |
| `get_trend_summary()` | Summarize trend classification results |
| `calculate_change_metrics()` | Compute change metrics across time periods |

### Part 7: Isochrones and 3D Geo Visualization

| Function | Purpose |
|----------|--------|
| `build_isochrone_request()` | Build provider-agnostic isochrone request |
| `get_isochrone()` | Fetch isochrone polygon from routing service |
| `ChartGenerator.create_3d_map()` | Render 3D intensity map from geo points |

### Part 8: H3 Hexagonal Spatial Indexing

| Function | Purpose |
|----------|--------|
| `h3_index_points()` | Assign H3 hex IDs to point rows |
| `h3_index_polygon()` | Get hex IDs covering a polygon |
| `h3_spatial_join()` | Approximate point-in-polygon via hex overlap |
| `h3_hex_to_boundary()` | Get boundary coordinates for a hex cell |
| `h3_resolution_for_area()` | Suggest resolution for a target area |

### Part 9: Boundary Providers

| Function | Purpose |
|----------|--------|
| `resolve_boundary_provider()` | Factory: TIGER for US, GADM for international |
| `CensusTIGERProvider` | US Census TIGER/Line boundary provider |
| `GADMProvider` | International boundaries via GADM |
| `BoundaryProvider.get_boundary()` | Fetch boundary geometry (uniform API) |
| `BoundaryProvider.list_levels()` | List available geographic levels |

**API Key**: Get a free Census API key at https://api.census.gov/data/key_signup.html
Set as `CENSUS_API_KEY` environment variable or pass `api_key` parameter.

### See Also

- **[15_Census_Demographics_Integration.ipynb](15_Census_Demographics_Integration.ipynb)** — Quick-start convenience functions
- **[20_Multi_Source_Spatial_Tabulation.ipynb](20_Multi_Source_Spatial_Tabulation.ipynb)** — Cross-source tabulation (Census × NCES × NLRB × RDH)

## Related
- Source: `siege_utilities/geo/providers/` (boundary_providers, census_geocoder)
- Tests: `tests/test_census_geocoder.py`, `tests/test_spatial_data*.py`
- Consolidates: `15_Census_Demographics_Integration`, parts of `26_International_Boundaries_GADM`
